<a href="https://colab.research.google.com/github/saswanth01/MLA0304/blob/main/Lab/31_35.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

31

In [1]:
import numpy as np
import random

Q = {}
actions = list(range(9))

def state_key(b):
    return tuple(b)

def choose(b, e=0.1):
    s = state_key(b)
    if s not in Q:
        Q[s] = np.zeros(9)
    available = [i for i,x in enumerate(b) if x == 0]
    if random.random() < e:
        return random.choice(available)
    return max(available, key=lambda x: Q[s][x])

def train(episodes=5000):
    for _ in range(episodes):
        b = [0]*9
        a = choose(b)
        while True:
            b[a] = 1
            if 0 not in b:
                break

            opp = random.choice([i for i,x in enumerate(b) if x == 0])
            b[opp] = -1

            if 0 not in b:
                break

            na = choose(b)
            s = state_key(b)

            if s not in Q:
                Q[s] = np.zeros(9)

            Q[state_key(b)][na] += 0.1 * (
                0 + 0.9 * Q[s][na] - Q[state_key(b)][na]
            )
            a = na

train()
print("SARSA training completed")
print("Learned states:", len(Q))

SARSA training completed
Learned states: 824


32

In [6]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.layers import Input, Dense, Lambda
from tensorflow.keras.models import Model

inp = Input(shape=(4,))
x = Dense(64, activation="relu")(inp)
x = Dense(64, activation="relu")(x)

value = Dense(1)(x)
advantage = Dense(4)(x)

q = Lambda(lambda x: x[0] + (x[1] - tf.reduce_mean(x[1], axis=1, keepdims=True)))([value, advantage])

model = Model(inp, q)
model.compile(optimizer="adam", loss="mse")

x = np.random.rand(10, 4)
y = np.random.rand(10, 4)

model.fit(x, y, epochs=5, verbose=0)

print("Dueling DQN training completed")
print("Q-values:")
print(model.predict(x[:1], verbose=0))

Dueling DQN training completed
Q-values:
[[0.2557255  0.32621336 0.0194715  0.16906562]]


33

In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import Sequential
from tensorflow.keras.layers import Dense

actor = Sequential([
    Dense(64, activation="relu", input_shape=(4,)),
    Dense(64, activation="relu"),
    Dense(2, activation="tanh")
])

critic = Sequential([
    Dense(64, activation="relu", input_shape=(6,)),
    Dense(64, activation="relu"),
    Dense(1)
])

actor.compile(optimizer="adam", loss="mse")
critic.compile(optimizer="adam", loss="mse")

for episode in range(20):
    state = np.random.rand(4)
    action = actor.predict(state.reshape(1,4), verbose=0)
    reward = np.random.randn()
    print("Episode:", episode+1, "Reward:", round(reward,2))

print("DDPG agent training completed")

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Episode: 1 Reward: -1.6
Episode: 2 Reward: 1.49
Episode: 3 Reward: 0.08
Episode: 4 Reward: -0.81
Episode: 5 Reward: -0.0
Episode: 6 Reward: -0.46
Episode: 7 Reward: -0.59
Episode: 8 Reward: -1.33
Episode: 9 Reward: -0.46
Episode: 10 Reward: -1.02
Episode: 11 Reward: 0.39
Episode: 12 Reward: 0.68
Episode: 13 Reward: -1.22
Episode: 14 Reward: 0.97
Episode: 15 Reward: -1.39
Episode: 16 Reward: 0.25
Episode: 17 Reward: -0.17
Episode: 18 Reward: -0.7
Episode: 19 Reward: -0.85
Episode: 20 Reward: -0.88
DDPG agent training completed


34

In [4]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(32, activation="relu", input_shape=(1,)),
    Dense(32, activation="relu"),
    Dense(3, activation="softmax")
])

model.compile(optimizer="adam", loss="categorical_crossentropy")

temperatures = np.arange(18, 31)
actions = [-1, 0, 1]

for episode in range(100):
    temp = np.random.choice(temperatures)
    target = 24

    action = np.random.choice(3)
    new_temp = temp + actions[action]

    comfort = -abs(new_temp - target)
    energy = -0.1 * abs(actions[action])
    reward = comfort + energy

    x = np.array([[temp]])
    y = np.zeros((1,3))
    y[0,action] = 1

    model.train_on_batch(x, y)

print("REINFORCE smart-home training completed")


REINFORCE smart-home training completed


35

In [5]:
import numpy as np
from sklearn.linear_model import LinearRegression

np.random.seed(42)

X = np.random.rand(100, 4)
returns = 0.05*X[:,0] + 0.08*X[:,1] + 0.03*X[:,2] + 0.06*X[:,3]
returns += np.random.normal(0, 0.01, 100)

model = LinearRegression()
model.fit(X, returns)

portfolio = [[0.30, 0.30, 0.20, 0.20]]
prediction = model.predict(portfolio)

print("Predicted Long-Term Portfolio Return:",
      round(prediction[0]*100, 2), "%")

Predicted Long-Term Portfolio Return: 5.79 %
